In [3]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # 0=all, 1=info, 2=warning, 3=error
from tensorflow.keras.utils import image_dataset_from_directory
import tensorflow as tf
IMG_SIZE = [224, 224]

def get_data(data_dir):
  train_ds = image_dataset_from_directory(
      data_dir,
      batch_size = 32,
      color_mode = 'rgb',
      image_size = IMG_SIZE,
      seed = 42,
      validation_split = 0.2,
      subset = 'training'
  )

  val_ds = image_dataset_from_directory(
      data_dir,
      batch_size = 32,
      color_mode = 'rgb',
      image_size = IMG_SIZE,
      seed = 42,
      validation_split = 0.2,
      subset = 'validation'
  )

  return train_ds, val_ds

def clean_data(train_ds, val_ds):
    # Apply EfficientNet preprocessing to image tensors
    train_ds = train_ds.map(
        lambda x, y: (tf.keras.applications.efficientnet.preprocess_input(x), y),
        num_parallel_calls=tf.data.AUTOTUNE
    )
    val_ds = val_ds.map(
        lambda x, y: (tf.keras.applications.efficientnet.preprocess_input(x), y),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    # Apply ignore_errors to gracefully skip corrupted files after preprocessing
    train_ds = train_ds.apply(tf.data.experimental.ignore_errors())
    val_ds   = val_ds.apply(tf.data.experimental.ignore_errors())

    # Cache and prefetch for performance
    train_ds = train_ds.cache("/tmp/train_cache").prefetch(tf.data.AUTOTUNE)
    val_ds = val_ds.cache("/tmp/train_cache").prefetch(tf.data.AUTOTUNE)

    return train_ds, val_ds

In [4]:
train_ds, val_ds = get_data('PetImages')
train_ds, val_ds = clean_data(train_ds, val_ds)

Found 24998 files belonging to 2 classes.
Using 19999 files for training.
Found 24998 files belonging to 2 classes.
Using 4999 files for validation.
Instructions for updating:
Use `tf.data.Dataset.ignore_errors` instead.


In [5]:
def build_model():

  from tensorflow.keras.applications import EfficientNetB0
  inputs = tf.keras.Input(shape = IMG_SIZE + [3])
  x = tf.keras.applications.efficientnet.preprocess_input(inputs)


  base_model = EfficientNetB0(input_tensor = x, weights = 'imagenet', include_top = False)

  for layer in base_model.layers :
    layer.trainable = False

  x = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
  x = tf.keras.layers.Dropout(0.3)(x)
  x = tf.keras.layers.Dense(1, activation = 'sigmoid')(x)
  model = tf.keras.Model(inputs = inputs, outputs = x)
  model.compile(
      optimizer = tf.keras.optimizers.Adam(learning_rate = 1e-4),
      loss = 'binary_crossentropy',
      metrics = ['accuracy']
  )

  return model

model = build_model()

16705208/16705208 [==============================] - 5s 0us/step


In [6]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    )
]


history = model.fit(train_ds, validation_data = val_ds, epochs = 10, callbacks = callbacks)

Epoch 1/10
      1/Unknown - 8s 8s/step - loss: 0.6923 - accuracy: 0.6250

I0000 00:00:1767776611.128969     136 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


     62/Unknown - 21s 222ms/step - loss: 0.5939 - accuracy: 0.7021

Corrupt JPEG data: 228 extraneous bytes before marker 0xd9


    242/Unknown - 52s 184ms/step - loss: 0.4381 - accuracy: 0.8483

...

    540/Unknown - 112s 193ms/step - loss: 0.3123 - accuracy: 0.9150

Corrupt JPEG data: 128 extraneous bytes before marker 0xd9


    606/Unknown - 120s 186ms/step - loss: 0.2940 - accuracy: 0.9219

Corrupt JPEG data: 1403 extraneous bytes before marker 0xd9


622/622 [==============================] - 181s 280ms/step - loss: 0.2902 - accuracy: 0.9235 - val_loss: 0.1304 - val_accuracy: 0.9826
Epoch 2/10
622/622 [==============================] - 112s 180ms/step - loss: 0.1060 - accuracy: 0.9825 - val_loss: 0.0733 - val_accuracy: 0.9873
Epoch 3/10
622/622 [==============================] - 116s 187ms/step - loss: 0.0704 - accuracy: 0.9847 - val_loss: 0.0533 - val_accuracy: 0.9891
Epoch 4/10
622/622 [==============================] - 121s 195ms/step - loss: 0.0541 - accuracy: 0.9881 - val_loss: 0.0432 - val_accuracy: 0.9900
Epoch 5/10
622/622 [==============================] - 121s 194ms/step - loss: 0.0462 - accuracy: 0.9887 - val_loss: 0.0372 - val_accuracy: 0.9905
Epoch 6/10
622/622 [==============================] - 119s 191ms/step - loss: 0.0413 - accuracy: 0.9887 - val_loss: 0.0334 - val_accuracy: 0.9909
Epoch 7/10
622/622 [==============================] - 118s 189ms/step - loss: 0.0385 - accuracy: 0.9883 - val_loss: 0.0307 - val_accura

In [7]:
model.save("cat_dog_efficientnet")

INFO:tensorflow:Assets written to: cat_dog_efficientnet/assets


INFO:tensorflow:Assets written to: cat_dog_efficientnet/assets


In [8]:
model.save("models/cat_dog.h5")

/usr/local/lib/python3.11/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
